<a href="https://colab.research.google.com/github/pavimunirajn2005-dev/Banking-FAQ-Assistant-final/blob/main/git_repository_analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Git Analyzer
### AI-Powered Repository Intelligence

A notebook version of the supplied Git Analyzer dashboard. Enter a public GitHub repository and run all cells.

In [1]:
REPOSITORY_URL = 'https://github.com/pavimunirajn2005-dev/Banking-FAQ-Assistant-final'
# Change this to any public GitHub repository URL.

In [2]:
import json, html
from datetime import datetime, timezone
from string import Template
from urllib.parse import urlparse
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
from IPython.display import HTML, display

def request(endpoint):
    req = Request('https://api.github.com' + endpoint, headers={'Accept':'application/vnd.github+json','User-Agent':'Git-Analyzer-Notebook'})
    with urlopen(req, timeout=20) as response:
        return json.loads(response.read().decode('utf-8'))

parsed = urlparse(REPOSITORY_URL.strip())
parts = [part for part in parsed.path.split('/') if part]
if parsed.netloc not in ('github.com','www.github.com') or len(parts) < 2:
    raise ValueError('Enter a public URL like https://github.com/pallets/flask')
owner, repository = parts[0], parts[1].removesuffix('.git')

try:
    repo = request(f'/repos/{owner}/{repository}')
    languages = request(f'/repos/{owner}/{repository}/languages')
    contributors = request(f'/repos/{owner}/{repository}/contributors?per_page=10')
    commits = request(f'/repos/{owner}/{repository}/commits?per_page=10')
    issues = request(f'/repos/{owner}/{repository}/issues?state=all&per_page=100')
    pulls = request(f'/repos/{owner}/{repository}/pulls?state=all&per_page=100')
except HTTPError as error:
    raise RuntimeError('Repository not found, private, or the GitHub API rate limit was reached.') from error
except URLError as error:
    raise RuntimeError('Could not connect to GitHub. Check your internet connection.') from error

print('Loaded:', repo['full_name'])

Loaded: pavimunirajn2005-dev/Banking-FAQ-Assistant-final


In [3]:
def number(value): return f'{value:,}'
def show_date(value): return datetime.fromisoformat(value.replace('Z','+00:00')).strftime('%d %b %Y') if value else '—'

open_issues = sum(x.get('state') == 'open' and 'pull_request' not in x for x in issues)
open_pulls = sum(x.get('state') == 'open' for x in pulls)
age = (datetime.now(timezone.utc) - datetime.fromisoformat(repo['updated_at'].replace('Z','+00:00'))).days
health = 50 + (10 if repo['stargazers_count'] >= 100 else 0) + (10 if repo['stargazers_count'] >= 1000 else 0) + (10 if len(contributors) >= 5 else 0) + (10 if age <= 30 else -10 if age > 365 else 0) - (25 if repo['archived'] else 0)
health = max(0, min(100, health))
health_title = 'Excellent Repository Health' if health >= 80 else 'Good Repository Health' if health >= 60 else 'Needs Attention'
total = sum(languages.values()) or 1
language_html = ''.join("<div class='language'><div><b>{}</b><span>{:.1%}</span></div><i><em style='width:{:.1%}'></em></i></div>".format(html.escape(k),v/total,v/total) for k,v in sorted(languages.items(),key=lambda x:x[1],reverse=True)[:8])
people_html = ''.join("<div class='person'><strong>{}</strong><div><b>{}</b><small>{} contributions</small></div></div>".format(html.escape(x.get('login','A')[0].upper()),html.escape(x.get('login','Anonymous')),number(x.get('contributions',0))) for x in contributors[:8])
commit_html = ''.join("<div class='commit'><b>{}</b><small>{} · {}</small></div>".format(html.escape(x['commit']['message'].splitlines()[0]),html.escape(x['commit']['author'].get('name','Unknown')),show_date(x['commit']['author'].get('date'))) for x in commits[:10])
details = [('Default Branch',repo.get('default_branch') or '—'),('Repository Size',number(repo.get('size',0))+' KB'),('Created',show_date(repo.get('created_at'))),('Last Updated',show_date(repo.get('updated_at'))),('License',(repo.get('license') or dict()).get('name') or 'Not specified'),('Archived','Yes' if repo['archived'] else 'No')]
detail_html = ''.join("<tr><td>{}</td><td>{}</td></tr>".format(a,html.escape(str(b))) for a,b in details)
dominant = max(languages,key=languages.get) if languages else 'not available'
insights = [('The repository is active and not marked as archived.' if not repo['archived'] else 'This repository is archived, so active development may have stopped.'),f'The dominant programming language is {dominant}.',('The contributor base is concentrated.' if len(contributors) <= 2 else 'The repository has a collaborative contributor presence.')]
recommendations = ['Review and prioritize unresolved issues.' if open_issues > 20 else 'Continue monitoring commits, issues, contributors, and pull requests together.']
note_html = ''.join("<div class='note'>{}</div>".format(html.escape(x)) for x in insights)
rec_html = ''.join("<div class='note rec'>{}</div>".format(html.escape(x)) for x in recommendations)

In [4]:
template = Template("""
<style>.dash{font-family:Arial,sans-serif;background:#f5f7fb;color:#172033;padding:28px;border-radius:18px}.top{background:#111827;color:white;padding:28px;border-radius:16px;margin-bottom:18px}.top h2{margin:0 0 8px}.top p{color:#cbd5e1}.top a{color:#93c5fd}.stats{display:grid;grid-template-columns:repeat(5,1fr);gap:12px;margin-bottom:18px}.stat,.card{background:white;border:1px solid #e5e7eb;border-radius:14px;padding:20px}.stat small,.card small{color:#667085;display:block}.stat b{font-size:27px;display:block;margin-top:7px}.grid{display:grid;grid-template-columns:1fr 1fr;gap:18px}.full{grid-column:1/-1}.card h3{margin-top:0}.health{display:flex;align-items:center;gap:20px}.score{border:10px solid #2563eb;width:122px;height:122px;border-radius:50%;display:flex;flex-direction:column;align-items:center;justify-content:center}.score b{font-size:31px}.language{margin:12px 0}.language div{display:flex;justify-content:space-between}.language i{display:block;background:#eef2f7;height:9px;border-radius:8px;margin-top:6px}.language em{display:block;background:#2563eb;height:100%;border-radius:8px}.person{display:flex;gap:11px;align-items:center;padding:10px 0;border-bottom:1px solid #edf0f4}.person>strong{background:#e0e7ff;color:#3730a3;border-radius:50%;height:39px;width:39px;text-align:center;padding-top:10px}.person b,.commit b{display:block}.info{width:100%;border-collapse:collapse}.info td{padding:11px 3px;border-bottom:1px solid #edf0f4}.info td:first-child{color:#667085;width:48%}.note{padding:13px;background:#f8fafc;border-left:4px solid #2563eb;border-radius:7px;margin:10px 0}.rec{background:#eff6ff;border-left-color:#1d4ed8}.commit{padding:11px 0;border-bottom:1px solid #edf0f4}@media(max-width:700px){.stats,.grid{grid-template-columns:1fr 1fr}}</style>
<div class='dash'><div class='top'><h2>⌘ Git Analyzer · $title</h2><p>$description</p><p><a href='$url' target='_blank'>View repository on GitHub →</a></p></div><div class='stats'><div class='stat'><small>Stars</small><b>$stars</b></div><div class='stat'><small>Forks</small><b>$forks</b></div><div class='stat'><small>Contributors</small><b>$contributors</b></div><div class='stat'><small>Open Issues</small><b>$issues</b></div><div class='stat'><small>Open PRs</small><b>$pulls</b></div></div><div class='grid'><section class='card'><h3>Repository Health</h3><div class='health'><div class='score'><b>$health</b><small>/100</small></div><div><b>$health_title</b><p>Score based on activity, stars, contributors, and archive status.</p></div></div></section><section class='card'><h3>Programming Languages</h3>$languages</section><section class='card'><h3>Repository Details</h3><table class='info'>$details</table></section><section class='card'><h3>Top Contributors</h3>$people</section><section class='card full'><h3>AI Repository Insights</h3>$insights</section><section class='card full'><h3>Recommendations</h3>$recommendations</section><section class='card full'><h3>Recent Commits</h3>$commits</section></div></div>
""")
page = template.safe_substitute(title=html.escape(repo['full_name']),description=html.escape(repo.get('description') or 'No repository description available.'),url=html.escape(repo['html_url']),stars=number(repo['stargazers_count']),forks=number(repo['forks_count']),contributors=number(len(contributors)),issues=number(open_issues),pulls=number(open_pulls),health=health,health_title=health_title,languages=language_html,details=detail_html,people=people_html,insights=note_html,recommendations=rec_html,commits=commit_html)
display(HTML(page))

Default Branch,main
Repository Size,91 KB
Created,11 Sep 2026
Last Updated,17 Sep 2026
License,Not specified
Archived,No
